# TRIDENT A1 / B0 preparation ? dense 4x, two epochs each
Only implementation and preflight are currently approved. **Training either row needs separate explicit approval.** Both launch flags remain false. No exports, official test evaluation or other ablations are included. See `docs/corrective_v1/implementation.md`.

Use a reviewed source ZIP or exact published Git commit, the existing Drive dataset ZIP and the existing Colab CUDA Python runtime. Setup preserves CUDA PyTorch and verifies source/data SHA256. Local Git changes must be published before GitHub mode can use them.


## 1. Settings
Keep both training flags false while preparing and reviewing GPU preflight. Choose a GPU runtime in Colab.


In [ ]:
from pathlib import Path
import hashlib, json, os, platform, shutil, subprocess, sys, zipfile
SOURCE_MODE = "archive"  # or "github" with an exact reviewed commit
GIT_URL = "https://github.com/Dhanush1756/TRIDENT-Depth-T69.git"
GIT_COMMIT = ""
SOURCE_SHA256 = ""  # copy the SHA256 of trident_A1_B0_source.zip
DATASET_SHA256 = "0e97b6d1a2ae7a83c4383566eee782dc04dc68048f9aff9590ebe0c7d9df1cd6"
DRIVE_ROOT = Path("/content/drive/MyDrive/TRIDENT-T69")
REPO = Path("/content/TRIDENT-Depth-T69")
CACHE = Path("/content/trident-nyu-converted")
TRAIN_A1_APPROVED = False
TRAIN_B0_APPROVED = False
EXPORT_APPROVED = False
RESUME = {"A1": "", "B0": ""}  # only epoch_001.pt after a disconnection
assert (3, 10) <= sys.version_info[:2] < (3, 14)
print(platform.python_version(), platform.platform())


## 2. Mount Drive
Upload `trident_A1_B0_source.zip` beside the existing `nyu_converted.zip` in `MyDrive/TRIDENT-T69`. No historical checkpoint is needed or modified.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
assert DRIVE_ROOT.is_dir(), f"Create {DRIVE_ROOT} and upload the archives first"


## 3. Verify and extract source
Use a fresh runtime for archive mode. GitHub mode requires a published reviewed commit containing these additions.


In [ ]:
if SOURCE_MODE == "github":
    assert len(GIT_COMMIT) == 40 and all(c in "0123456789abcdefABCDEF" for c in GIT_COMMIT), "Set the reviewed full Git commit"
    if not REPO.exists():
        subprocess.run(["git", "clone", GIT_URL, str(REPO)], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", GIT_COMMIT], check=True)
    actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
    assert actual.lower() == GIT_COMMIT.lower()
    assert not subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip(), "Checkout has local modifications"
    SOURCE_ID = actual
elif SOURCE_MODE == "archive":
    archive = DRIVE_ROOT / "trident_A1_B0_source.zip"
    assert len(SOURCE_SHA256) == 64, "Set source archive SHA256"
    digest = hashlib.sha256()
    with archive.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    assert digest.hexdigest() == SOURCE_SHA256
    assert not REPO.exists(), "Use a fresh runtime or GitHub mode for an existing checkout"
    with zipfile.ZipFile(archive) as bundle:
        names = set()
        for item in bundle.infolist():
            name = item.filename
            target = (REPO / name).resolve()
            assert "\\" not in name and target.is_relative_to(REPO.resolve())
            assert name not in names and (item.external_attr >> 16) & 0o170000 != 0o120000
            names.add(name)
        bundle.extractall(REPO)
    SOURCE_ID = SOURCE_SHA256
else:
    raise ValueError("SOURCE_MODE must be github or archive")
assert (REPO / "trident_depth/training/colab.py").is_file(), "Reviewed Colab changes are missing"
os.chdir(REPO)


## 4. Existing runtime, dependencies and GPU detection
Use the existing Colab Python runtime directly, preserving its CUDA-enabled PyTorch. Only additional requirements are installed; any dependency plan touching PyTorch or its CUDA stack is rejected. The repository installs with dependency resolution disabled. Fresh subprocesses use the updated packages without creating a virtual environment. Record exact Python, torch, CUDA, package versions and GPU, and run the complete project test suite after installation. Resume requires the same software fingerprint; hardware changes may still affect numerical reproducibility.

In [ ]:
# Use Colab's existing runtime directly: no venv or ensurepip.
import tempfile
PY = sys.executable
print("Colab Python:", PY)
torch_probe = """
import json, torch
print(json.dumps(dict(version=str(torch.__version__), cuda=torch.version.cuda,
 location=torch.__file__, cuda_available=torch.cuda.is_available())))
"""
TORCH_BEFORE = json.loads(subprocess.check_output([str(PY), "-c", torch_probe], text=True))
assert TORCH_BEFORE["cuda_available"], "Select a GPU runtime; Colab CUDA PyTorch must already be available"

# Resolve only the additional requirements first. Reject any plan that would
# touch PyTorch or its CUDA stack, then install exactly that plan without resolving
# further dependencies. This cannot pull in a CPU-only torch wheel.
with tempfile.TemporaryDirectory(prefix="trident-pip-") as temporary:
    report = Path(temporary) / "install-plan.json"
    subprocess.run([str(PY), "-m", "pip", "install", "--dry-run",
                    "--only-binary=:all:", "--report", str(report),
                    "-r", str(REPO / "requirements-colab.txt")], check=True)
    plan = json.loads(report.read_text())["install"]
    protected = {"torch", "torchvision", "torchaudio", "torchtext", "triton", "pytorch-triton"}
    for item in plan:
        name = item["metadata"]["name"].lower().replace("_", "-")
        assert name not in protected and not name.startswith("nvidia-"), f"Refusing to replace Colab GPU package: {name}"
    urls = [item["download_info"]["url"] for item in plan]
    if urls:
        subprocess.run([str(PY), "-m", "pip", "install", "--no-deps",
                        "--only-binary=:all:", *urls], check=True)
subprocess.run([str(PY), "-m", "pip", "install", "--no-deps", "--no-build-isolation", "-e", str(REPO)], check=True)
TORCH_AFTER = json.loads(subprocess.check_output([str(PY), "-c", torch_probe], text=True))
assert TORCH_AFTER == TORCH_BEFORE, "Colab PyTorch/CUDA changed during setup"
probe = """
import json, platform, torch, numpy, scipy, cv2, PIL, h5py
assert (2, 6) <= tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) < (3, 0)
print(json.dumps(dict(python=platform.python_version(), torch=str(torch.__version__),
 cuda=torch.version.cuda, numpy=numpy.__version__, scipy=scipy.__version__,
 opencv=cv2.__version__, pillow=PIL.__version__, h5py=h5py.__version__,
 gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
 gpu_memory_bytes=torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None)))
"""
ENVIRONMENT = json.loads(subprocess.check_output([str(PY), "-c", probe], text=True))
print(json.dumps(ENVIRONMENT, indent=2))
assert ENVIRONMENT["gpu"], "No CUDA GPU available: select a GPU runtime. Do not run the pilot on CPU."
# Complete project suite after installation; CPU tests do not train a model.
subprocess.run([str(PY), "-m", "pytest", "-q"], check=True,
               env=dict(os.environ, CUDA_VISIBLE_DEVICES=""))


## 5. Stage and audit the official dataset
Copy/extract from the uploaded ZIP into temporary Colab storage once per runtime. Archive SHA256 and the approved manifest SHA256 are checked. Preserve 672 training / 123 validation / 654 locked test records and their scene assignments. Test pixels are not loaded by training or validation.

In [ ]:
assert len(DATASET_SHA256) == 64, "Set the dataset archive SHA256 printed on Windows"
stage = """
import sys, json
from pathlib import Path
from trident_depth.training.colab import stage_archive, audit_pilot_dataset
archive, cache, expected = sys.argv[1:]
if not Path(cache).exists():
    stage_archive(archive, cache, expected)
assert (Path(cache) / '.archive_sha256').read_text() == expected, 'Cache belongs to another archive'
print(json.dumps(audit_pilot_dataset(Path(cache) / 'manifest_development.json'), indent=2))
"""
subprocess.run([str(PY), "-c", stage, str(DRIVE_ROOT / "nyu_converted.zip"),
                str(CACHE), DATASET_SHA256], check=True)
MANIFEST = CACHE / "manifest_development.json"


## 6. Paired GPU preflight ? zero optimizer steps
Run synthetic and two full-resolution **training** frames per row, identity and paired-RNG checks, first-step gradient diagnostics, calibration probes for A1, and temporary checkpoint/resume round-trips. B0 never executes its uncertainty head; its initial reconstruction-backbone gradients are correctly zero. The official test pixels stay locked. Failure here blocks launching either row.


In [ ]:
assert EXPORT_APPROVED is False
preflight_command = [str(PY), "-m", "trident_depth.training.corrective_preflight",
    "--configs", str(REPO / "configs/corrective_v1"),
    "--manifest", str(MANIFEST), "--device", "cuda"]
PREFLIGHT = json.loads(subprocess.check_output(preflight_command, text=True))
assert PREFLIGHT["parameters"] == 473059 and PREFLIGHT["optimizer_steps"] == 0
assert PREFLIGHT["cuda_verified"] and PREFLIGHT["matching_non_head_tensors_and_constructor_rng"]
assert PREFLIGHT["official_test_pixels_loaded"] == 0
print(json.dumps(PREFLIGHT, indent=2))
from datetime import datetime, timezone
PREPARATION_DIR = DRIVE_ROOT / "preparation" / "A1_B0"
PREPARATION_DIR.mkdir(parents=True, exist_ok=True)
PREFLIGHT_FILE = PREPARATION_DIR / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + "_gpu_preflight.json")
with PREFLIGHT_FILE.open("x") as stream:
    json.dump(PREFLIGHT, stream, indent=2)
EXPERIMENTS = {row: REPO / f"configs/corrective_v1/{row}.json" for row in ("A1", "B0")}
RECORDS = {}
RUN_DIRS = {}
ENV_FILES = {}
for row, experiment in EXPERIMENTS.items():
    cfg = json.loads(experiment.read_text())
    run_id = f"corrective_v1_{row}_dense4_seed69"
    RUN_DIRS[row] = DRIVE_ROOT / "runs" / run_id
    ENV_FILES[row] = DRIVE_ROOT / "runs" / (run_id + "_environment.json")
    RECORDS[row] = dict(source_id=SOURCE_ID, environment=ENVIRONMENT,
        dataset_archive_sha256=DATASET_SHA256,
        manifest_sha256=PREFLIGHT["split_audit"]["manifest_sha256"],
        paired_initial_tensor_sha256=PREFLIGHT["paired_initial_tensor_sha256"],
        experiment_sha256=hashlib.sha256(experiment.read_bytes()).hexdigest(),
        pipeline_sha256=hashlib.sha256((experiment.parent/cfg["pipeline_config"]).read_bytes()).hexdigest(),
        boundary_evaluation=PREFLIGHT["boundary_evaluation"])
    if ENV_FILES[row].exists():
        assert json.loads(ENV_FILES[row].read_text()) == RECORDS[row], "Restore the exact source/environment/data before resume"
print("Preparation complete. Training remains disabled pending separate approval.")


## 7. Gated launch helper
Do not enable a flag until that experiment has explicit training approval. Each row is a fresh independent seed-69 initialization, not a continuation of another pilot. The frozen runner rejects changes to the two-epoch budget or objective. It validates every epoch, saves complete Adam/scheduler/RNG/history/provenance and stops after epoch two. Both cells below safely skip by default.

For disconnection recovery: use the same source/runtime/device, directories and environment record. Set only the approved row's RESUME value to `epoch_001.pt`; it resumes epoch two without resetting the learned head. A partially completed epoch replays from the last completed epoch. If there is no completed checkpoint, preserve the interrupted folder and request a reviewed fresh run path. Never resume epoch two to extend training. No export or test evaluation follows.


In [ ]:
def launch_control(row, approved):
    if not approved:
        print(f"{row}: training disabled; no optimizer steps.")
        return
    assert EXPORT_APPROVED is False and PREFLIGHT["cuda_verified"]
    run_dir, env_file = RUN_DIRS[row], ENV_FILES[row]
    resume = RESUME[row]
    if resume:
        assert resume == "epoch_001.pt" and (run_dir/resume).is_file()
        assert env_file.is_file(), "Original provenance is required for resume"
    else:
        assert not run_dir.exists(), "Run exists; do not overwrite or restart it"
    env_file.parent.mkdir(parents=True, exist_ok=True)
    if not env_file.exists():
        with env_file.open("x") as stream:
            json.dump(RECORDS[row], stream, indent=2)
        env_file.with_suffix(".freeze.txt").write_text(
            subprocess.check_output([str(PY), "-m", "pip", "freeze"], text=True))
    command = [str(PY), "-m", "trident_depth.training.corrective_train",
        "--experiment", str(EXPERIMENTS[row]), "--manifest", str(MANIFEST),
        "--output", str(run_dir), "--device", "cuda", "--record-boundary-f1",
        "--provenance", str(env_file), "--execute-training"]
    if resume:
        command += ["--resume", str(run_dir/resume)]
    subprocess.run(command, check=True)


## 8. A1 ? disabled pending training approval
Exact-zero residual, alpha=1, L_J.


In [ ]:
launch_control("A1", TRAIN_A1_APPROVED)


## 9. B0 ? disabled pending training approval
Same initial tensors and shuffle, alpha=1, L_R. Uncertainty head unavailable; boundary head unsupervised in both rows.


In [ ]:
launch_control("B0", TRAIN_B0_APPROVED)


## Stop
Do not launch further runs, export weights or evaluate the official test split. Download the preparation JSON for review now. After separately approved training, preserve each run folder (both epoch checkpoints, history, best pointer, environment JSON and package freeze) for a new paired validation audit. Loss totals across L_J and L_R are not quality rankings.
